# gesn-search: разбор ошибок и значимость

Берёт дообученную модель из вывода ноутбука `kaggle_pipeline` (файл `e5-gesn.zip`) и считает:
- ранги и top-5 каждого подхода на каждый тестовый запрос (`per_query.jsonl`);
- парные сравнения подходов: бутстреп-интервал различия и тест Макнемара (`significance.md`).

**Перед запуском:**
1. Справа **Add Input** → вкладка **Your Work** (или поиск по названию) → выбрать ноутбук с прошлым прогоном → **Add**.
2. Session options: Accelerator — **GPU T4 x2**, Internet — **On**.

Затем **Run All** (~5 минут).

In [ ]:
!rm -rf /tmp/gesn-search && git clone -q https://github.com/malma0/gesn-search /tmp/gesn-search
%cd /tmp/gesn-search
!pip install -q -e . --no-deps
!pip install -q "rank-bm25>=0.2.2" "pymorphy3>=2.0" "sentence-transformers>=5.0"

In [ ]:
import glob, zipfile

zips = glob.glob("/kaggle/input/**/e5-gesn.zip", recursive=True)
assert zips, "Не найден e5-gesn.zip — добавь вывод прошлого ноутбука через Add Input"
zipfile.ZipFile(zips[0]).extractall("models/e5-gesn")
print("модель:", zips[0])

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m gesn.analyze --methods bm25 e5 ft=models/e5-gesn

In [ ]:
!cat results/per_query.jsonl